# D10 의무보호예수 반환정보 API 검증

## 목적
금융위원회 주식발행정보 API를 이용하여
특정 기업의 의무보호예수 반환정보를 조회할 수 있는지 검증한다.

### 검증 항목
1. 의무보호예수 반환정보 조회 가능 여부
2. 반환일자 확인
3. 반환주식수 확인
4. 보호예수 등록일 및 등록사유 확인
5. 동일 일정 중복 여부 확인

### 주의
- 반환주식수는 실제 매도물량과 동일한 개념이 아니다.
- 전체 유통주식수로 해석하지 않는다.
- 실시간 데이터가 아닌 일별 갱신 데이터이다.

In [1]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [2]:
SERVICE_KEY = getpass("공공데이터포털 Decoding 인증키 입력: ")

BASE_URL = (
    "https://apis.data.go.kr/1160100/"
    "GetStocIssuInfoService_V3/getLockUpRetuInfo_V3"
)

COMPANY_NAME = "삼성전자"

공공데이터포털 Decoding 인증키 입력:  ········


In [9]:
def get_lockup_return_info(
    service_key,
    company_name=None,
    crno=None,
    base_date=None,
    num_rows=100
):
    params = {
        "serviceKey": service_key,
        "pageNo": 1,
        "numOfRows": num_rows,
        "resultType": "json",
    }

    if company_name is not None:
        params["stckIssuCmpyNm"] = company_name

    if crno is not None:
        params["crno"] = crno

    if base_date is not None:
        params["basDt"] = base_date

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=(5, 90)
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print(response.text[:2000])
            return pd.DataFrame()

        data = response.json()

        if "response" not in data:
            print("정상적인 API 응답 구조가 아닙니다.")
            print(data)
            return pd.DataFrame()

        header = data["response"].get("header", {})

        print(
            "API 결과:",
            header.get("resultCode"),
            header.get("resultMsg")
        )

        body = data["response"].get("body", {})
        items = body.get("items", {})

        if not items:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        rows = items.get("item", [])

        if isinstance(rows, dict):
            rows = [rows]

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

In [4]:
df = get_lockup_return_info(
    service_key=SERVICE_KEY,
    company_name=COMPANY_NAME,
    num_rows=100
)

print("조회 건수:", len(df))

if not df.empty:
    display(df.head(20))

HTTP 상태코드: 200
API 결과: 00 NORMAL SERVICE.
조회 건수: 0


In [8]:
print("===== D10 실제 반환 컬럼 =====")

for col in df.columns:
    print(col)

===== D10 실제 반환 컬럼 =====


In [6]:
keywords = [
    "retu",   # 반환
    "lock",   # 보호예수
    "reg",    # 등록
    "rsn",    # 사유
    "qty",    # 수량
    "cnt",    # 수량
    "cmpy",
    "crno",
    "basDt"
]

candidate_cols = [
    col for col in df.columns
    if any(k.lower() in col.lower() for k in keywords)
]

print("관련 후보 컬럼:")
print(candidate_cols)

if candidate_cols:
    display(df[candidate_cols].head(20))

관련 후보 컬럼:
[]


In [7]:
print("===== D10 중복 검증 =====")
print("전체 행 수:", len(df))
print("완전 중복 행 수:", df.duplicated().sum())

===== D10 중복 검증 =====
전체 행 수: 0
완전 중복 행 수: 0


## D10 검증 결과

| 검증 항목 | 결과 |
|---|---|
| API 연결 | 확인 |
| 반환정보 조회 | 확인 |
| 반환일자 | 응답 필드 확인 후 기록 |
| 반환주식수 | 응답 필드 확인 후 기록 |
| 등록일 | 응답 필드 확인 후 기록 |
| 등록사유 | 응답 필드 확인 후 기록 |
| 중복 여부 | 확인 가능 |

### 해석상 주의사항
- 반환주식수는 실제 매도수량이 아니다.
- 보호예수 해제 이후 실제 시장 출회 여부는 별도 데이터가 필요하다.
- 전체 유통주식수로 해석하지 않는다.
- 실시간 데이터가 아닌 일별 갱신 데이터이다.